# Classical Decomposition

Classical decomposition is the oldest and simplest method for splitting a time
series into its structural components: **trend**, **seasonality**, and
**remainder** (also called residual or irregular). Despite its limitations, it
provides the conceptual foundation for every modern decomposition technique.

This notebook walks through the algorithm step by step, then uses the
statsmodels implementation, and finishes with a discussion of why classical
decomposition is no longer recommended for serious work.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

# Plotting defaults
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
# --- Airline Passengers (monthly, 1949-1960) ---
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]

# --- Energy Production (monthly, 1970-2018) ---
energy = pd.read_csv(
    DATA_DIR / "EnergyProduction.csv",
    index_col="DATE",
    parse_dates=True,
)
energy.index.freq = "MS"
energy.columns = ["Production"]

print("Airline shape:", airline.shape)
print("Energy shape:", energy.shape)
airline.head()

## The Two Models

Classical decomposition assumes that an observed time series $y_t$ can be
broken into three components. There are two variants:

### Additive Decomposition

$$y_t = T_t + S_t + R_t$$

where
- $T_t$ = trend-cycle component
- $S_t$ = seasonal component
- $R_t$ = remainder (residual)

**When to use:** the seasonal variation is roughly **constant** regardless of
the level of the series. The seasonal swings have the same absolute magnitude
whether the series is high or low.

### Multiplicative Decomposition

$$y_t = T_t \times S_t \times R_t$$

**When to use:** the seasonal variation **grows proportionally** with the level
of the series. This is extremely common in economic and business data (e.g.,
airline passengers, retail sales).

> **Tip:** A quick diagnostic is to plot the series and check whether the
> seasonal peaks and troughs fan out over time. If they do, use multiplicative.

In [ ]:
# Visual diagnostic: airline passengers clearly fans out -> multiplicative
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(airline)
axes[0].set_title("Airline Passengers\n(seasonal amplitude grows → multiplicative)")
axes[0].set_ylabel("Thousands of Passengers")

axes[1].plot(energy)
axes[1].set_title("Energy Production\n(seasonal amplitude roughly constant → additive)")
axes[1].set_ylabel("Energy Index")

plt.tight_layout()
plt.show()

## The Algorithm (Additive, Step by Step)

We will manually implement classical additive decomposition for the energy
production series, which has monthly data ($m = 12$).

### Step 1: Estimate the Trend with a $2 \times 12$-MA

For even-order seasonality ($m = 12$), the standard approach is a
**$2 \times m$ moving average**: first apply a 12-period MA, then smooth the
result with a 2-period MA. This is equivalent to a **weighted** 12-period MA
where the first and last observations get half weight.

$$\hat{T}_t = \frac{1}{2}\left(\frac{1}{12}\sum_{j=-6}^{5} y_{t+j} + \frac{1}{12}\sum_{j=-5}^{6} y_{t+j}\right)$$

In [ ]:
y = energy["Production"].copy()

# 2x12-MA: 12-period centred MA followed by 2-period MA
ma_12 = y.rolling(window=12, center=True).mean()
trend = ma_12.rolling(window=2, center=True).mean()

fig, ax = plt.subplots()
ax.plot(y, label="Original", alpha=0.5)
ax.plot(trend, label="Trend (2×12-MA)", linewidth=2.5, color="tab:red")
ax.set_title("Step 1: Estimate the Trend — Energy Production")
ax.set_ylabel("Energy Index")
ax.legend()
plt.tight_layout()
plt.show()

print(f"NaN count in trend: {trend.isna().sum()} (first and last {12//2} observations)")

### Step 2: Detrend the Series

For the additive model, we subtract the trend from the original series:

$$y_t - \hat{T}_t = S_t + R_t$$

(For multiplicative, we would divide: $y_t \,/\, \hat{T}_t$)

In [ ]:
detrended = y - trend

fig, ax = plt.subplots()
ax.plot(detrended, color="tab:green", alpha=0.7)
ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
ax.set_title("Step 2: Detrended Series (y - Trend)")
ax.set_ylabel("Detrended Value")
plt.tight_layout()
plt.show()

### Step 3: Estimate the Seasonal Component

Average the detrended values for each calendar month (January values together,
February values together, etc.) to get the seasonal indices $\hat{S}_t$. Then
tile these 12 values across the full series.

In [ ]:
# Average detrended value for each month
monthly_avg = detrended.groupby(detrended.index.month).mean()

# Adjust so seasonal indices sum to zero (additive requirement)
monthly_avg -= monthly_avg.mean()

print("Seasonal indices (one per month):")
print(monthly_avg.round(2))
print(f"\nSum of seasonal indices: {monthly_avg.sum():.6f} (should be ≈ 0)")

In [ ]:
# Map seasonal indices back to the full series
seasonal = y.index.month.map(monthly_avg)
seasonal = pd.Series(seasonal, index=y.index, name="Seasonal")

fig, ax = plt.subplots()
ax.plot(seasonal, color="tab:purple")
ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
ax.set_title("Step 3: Seasonal Component (repeats every 12 months)")
ax.set_ylabel("Seasonal Index")
plt.tight_layout()
plt.show()

### Step 4: Compute the Remainder

$$\hat{R}_t = y_t - \hat{T}_t - \hat{S}_t$$

Ideally, the remainder should look like white noise — no patterns, no trends.

In [ ]:
remainder = y - trend - seasonal

fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)

axes[0].plot(y, color="tab:blue")
axes[0].set_ylabel("Observed")
axes[0].set_title("Manual Classical Additive Decomposition — Energy Production")

axes[1].plot(trend, color="tab:red")
axes[1].set_ylabel("Trend")

axes[2].plot(seasonal, color="tab:purple")
axes[2].set_ylabel("Seasonal")

axes[3].plot(remainder, color="tab:green")
axes[3].axhline(0, color="grey", linestyle="--", linewidth=0.8)
axes[3].set_ylabel("Remainder")
axes[3].set_xlabel("Date")

plt.tight_layout()
plt.show()

## Using `seasonal_decompose()` in statsmodels

The `seasonal_decompose()` function from statsmodels implements exactly the
algorithm we just did manually. It returns an object with four attributes:
`.observed`, `.trend`, `.seasonal`, `.resid`.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

result_add = seasonal_decompose(energy["Production"], model="additive", period=12)
result_add.plot()
plt.suptitle("seasonal_decompose — Additive Model (Energy Production)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Accessing individual components
print("Component shapes:")
print(f"  observed : {result_add.observed.shape}")
print(f"  trend    : {result_add.trend.shape}  (NaN at edges)")
print(f"  seasonal : {result_add.seasonal.shape}")
print(f"  resid    : {result_add.resid.shape}  (NaN at edges)")

print(f"\nNaN count in trend: {result_add.trend.isna().sum()}")
print(f"NaN count in resid: {result_add.resid.isna().sum()}")

In [ ]:
# Verify: observed = trend + seasonal + resid
reconstructed = result_add.trend + result_add.seasonal + result_add.resid
diff = (result_add.observed - reconstructed).dropna()
print(f"Max reconstruction error: {diff.abs().max():.2e}")
print("(Should be essentially zero — just floating-point noise)")

## Additive vs Multiplicative Comparison

Airline passengers exhibit **growing seasonal amplitude**, making multiplicative
decomposition the better choice. Let's compare both models and inspect the
residuals.

In [ ]:
result_add_air = seasonal_decompose(airline["Passengers"], model="additive", period=12)
result_mul_air = seasonal_decompose(airline["Passengers"], model="multiplicative", period=12)

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)

# Additive (left column)
axes[0, 0].plot(result_add_air.observed)
axes[0, 0].set_ylabel("Observed")
axes[0, 0].set_title("Additive")

axes[1, 0].plot(result_add_air.trend, color="tab:red")
axes[1, 0].set_ylabel("Trend")

axes[2, 0].plot(result_add_air.seasonal, color="tab:purple")
axes[2, 0].set_ylabel("Seasonal")

axes[3, 0].plot(result_add_air.resid, color="tab:green")
axes[3, 0].axhline(0, color="grey", linestyle="--", linewidth=0.8)
axes[3, 0].set_ylabel("Remainder")

# Multiplicative (right column)
axes[0, 1].plot(result_mul_air.observed)
axes[0, 1].set_title("Multiplicative")

axes[1, 1].plot(result_mul_air.trend, color="tab:red")

axes[2, 1].plot(result_mul_air.seasonal, color="tab:purple")

axes[3, 1].plot(result_mul_air.resid, color="tab:green")
axes[3, 1].axhline(1, color="grey", linestyle="--", linewidth=0.8)

plt.suptitle("Airline Passengers — Additive vs Multiplicative Decomposition", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Compare residuals: multiplicative residuals should be more random
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(result_add_air.resid, color="tab:green", alpha=0.7)
axes[0].axhline(0, color="grey", linestyle="--", linewidth=0.8)
axes[0].set_title("Additive Residuals")
axes[0].set_ylabel("Residual")

axes[1].plot(result_mul_air.resid, color="tab:orange", alpha=0.7)
axes[1].axhline(1, color="grey", linestyle="--", linewidth=0.8)
axes[1].set_title("Multiplicative Residuals")
axes[1].set_ylabel("Residual")

plt.suptitle("Residual Comparison — Airline Passengers", fontsize=13)
plt.tight_layout()
plt.show()

print("Additive residual std:       ", result_add_air.resid.dropna().std().round(2))
print("Multiplicative residual std: ", result_mul_air.resid.dropna().std().round(4))
print("\nThe additive residuals show a clear pattern (variance grows over time),")
print("confirming that the multiplicative model is the better fit.")

### Residual Histograms

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(result_add_air.resid.dropna(), bins=20, color="tab:green", edgecolor="white")
axes[0].set_title("Additive Residuals")
axes[0].set_xlabel("Residual")

axes[1].hist(result_mul_air.resid.dropna(), bins=20, color="tab:orange", edgecolor="white")
axes[1].set_title("Multiplicative Residuals")
axes[1].set_xlabel("Residual")

plt.suptitle("Distribution of Residuals", fontsize=13)
plt.tight_layout()
plt.show()

print("The multiplicative residuals are more symmetrically distributed around 1.0,")
print("indicating a better decomposition.")

## Energy Production — Additive Decomposition

For comparison, let's apply additive decomposition to the energy production
data, which has roughly constant seasonal amplitude.

In [ ]:
result_energy = seasonal_decompose(energy["Production"], model="additive", period=12)
result_energy.plot()
plt.suptitle("Energy Production — Additive Decomposition", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Seasonal indices for energy production
seasonal_indices = result_energy.seasonal.iloc[:12]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, 13), seasonal_indices.values, color="tab:purple", edgecolor="white")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
ax.set_title("Seasonal Indices — Energy Production")
ax.set_ylabel("Seasonal Effect")
plt.tight_layout()
plt.show()

print("Energy production peaks in summer (Jul-Aug) and winter (Jan),")
print("with lows in spring (Apr-May) and autumn (Oct-Nov).")

## Limitations of Classical Decomposition

Classical decomposition was developed in the 1920s and remains a useful
teaching tool, but it has several important shortcomings. The following
limitations are summarized from Hyndman & Athanasopoulos,
*Forecasting: Principles and Practice* (FPP3).

### 1. Missing Edge Values

The trend estimate uses a centred moving average, so the first and last $m/2$
observations have no trend (and therefore no remainder). For monthly data,
that means the first 6 and last 6 values are lost.

In [ ]:
# Demonstrate missing edge values
trend_air = result_add_air.trend

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(airline["Passengers"], label="Observed", alpha=0.5)
ax.plot(trend_air, label="Trend", linewidth=2, color="tab:red")

# Highlight missing regions
ax.axvspan(airline.index[0], airline.index[5], alpha=0.2, color="red", label="No trend estimate")
ax.axvspan(airline.index[-6], airline.index[-1], alpha=0.2, color="red")

ax.set_title("Limitation 1: Missing Edge Values")
ax.set_ylabel("Passengers")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Total observations: {len(airline)}")
print(f"Observations with trend estimate: {trend_air.notna().sum()}")
print(f"Missing: {trend_air.isna().sum()} ({trend_air.isna().sum()/len(airline)*100:.1f}%)")

### 2. Fixed Seasonal Pattern

Classical decomposition assumes that the seasonal component is **identical**
every year. In reality, seasonal patterns often evolve over time — for
example, holiday shopping seasons tend to start earlier each year.

In [ ]:
# Show that the seasonal component repeats exactly
seasonal_air = result_add_air.seasonal

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(seasonal_air, color="tab:purple")
ax.set_title("Limitation 2: Seasonal Component is Identical Every Year")
ax.set_ylabel("Seasonal Effect")
plt.tight_layout()
plt.show()

# Prove it: check unique seasonal patterns
unique_patterns = seasonal_air.values.reshape(-1, 12)
print("All rows identical?", np.allclose(unique_patterns[0], unique_patterns[1:]))

### 3. Not Robust to Outliers

A single outlier can distort both the trend and the seasonal component,
because the moving average gives equal weight to all observations.

In [ ]:
# Inject an artificial outlier
airline_outlier = airline["Passengers"].copy()
outlier_idx = "1958-06-01"
airline_outlier.loc[outlier_idx] = airline_outlier.loc[outlier_idx] + 250  # huge spike

result_clean = seasonal_decompose(airline["Passengers"], model="additive", period=12)
result_dirty = seasonal_decompose(airline_outlier, model="additive", period=12)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(result_clean.trend, label="Without outlier", linewidth=2)
axes[0].plot(result_dirty.trend, label="With outlier", linewidth=2, linestyle="--")
axes[0].set_title("Limitation 3: Outlier Distorts the Trend")
axes[0].set_ylabel("Trend")
axes[0].legend()

axes[1].plot(result_clean.seasonal.iloc[:12], label="Without outlier", marker="o")
axes[1].plot(result_dirty.seasonal.iloc[:12], label="With outlier", marker="s", linestyle="--")
axes[1].set_title("Outlier Also Distorts the Seasonal Component")
axes[1].set_ylabel("Seasonal")
axes[1].legend()

plt.tight_layout()
plt.show()

### 4. Over-Smoothing and Other Issues

The remaining limitations of classical decomposition:

- **Over-smooths rapid changes**: The moving average is a linear smoother that
  cannot capture sudden shifts in the trend (e.g., a recession).
- **Only handles additive or multiplicative**: It cannot model more complex
  relationships between components.

As Hyndman & Athanasopoulos state in FPP3:

> "*Classical decomposition methods are not recommended — there are now several
> much better methods for decomposing a time series into components.*"

## Summary

| Aspect | Additive | Multiplicative |
|---|---|---|
| **Formula** | $y_t = T_t + S_t + R_t$ | $y_t = T_t \times S_t \times R_t$ |
| **Use when** | Seasonal amplitude is constant | Seasonal amplitude grows with level |
| **Residual baseline** | 0 | 1 |
| **Trend estimation** | $2 \times m$ moving average | Same |
| **Detrending** | $y_t - \hat{T}_t$ | $y_t \,/\, \hat{T}_t$ |

**Key limitations of classical decomposition:**
1. Missing values at edges (first/last $m/2$ observations)
2. Assumes a fixed (repeating) seasonal pattern
3. Not robust to outliers
4. Over-smooths rapid trend changes
5. Not recommended for production use — use STL or X-13ARIMA instead

In the next notebook, we will explore **STL decomposition**, which addresses
all of these shortcomings.